# PSM Experiment — Kaggle Notebook

**Predictive Semantic Memory (PSM)** experiment runner for Kaggle.

This notebook:
1. Loads a Kaggle-hosted LLM (no external API key needed)
2. Runs a **smoke test** (synthetic questions) to validate the pipeline end-to-end
3. Runs the **full TriviaQA experiment** (smoke + pilot profiles)
4. Saves **all results as human-readable CSV files** ready to push to GitHub


## 1 · Install required packages

In [ ]:
%%capture install_output
import subprocess, sys
PACKAGES = ["faiss-cpu", "sentence-transformers", "rank-bm25", "datasets", "rouge-score", "nltk"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *PACKAGES])
print("✓ All packages installed.")


In [ ]:
import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
print("✓ NLTK data ready.")


## 1b · Clone PSM source code from GitHub

In [ ]:
import subprocess
from pathlib import Path
REPO_URL = "https://github.com/KausikVP30/PSM_Experiment_Research_Setup_Updated.git"
CLONE_DIR = Path("/kaggle/working/psm_repo")
if not CLONE_DIR.exists():
    print(f"Cloning PSM repo from {REPO_URL} ...")
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(CLONE_DIR)])
    print("✓ PSM repo cloned.")
else:
    print("✓ PSM repo already cloned.")


## 2 · Configuration — Auto-detect model path

In [ ]:
import os, glob
from pathlib import Path

# Auto-detect the model by scanning /kaggle/input for a directory containing config.json
# Kaggle may mount models at /kaggle/input/models/... or /kaggle/input/qwen-lm/...
KAGGLE_MODEL_PATH = None

# First try well-known static paths
STATIC_CANDIDATES = [
    "/kaggle/input/qwen-lm/qwen2/Transformers/7b-instruct-gptq-int4/1",
    "/kaggle/input/qwen-lm/qwen2/transformers/7b-instruct-gptq-int4/1",
]
for c in STATIC_CANDIDATES:
    if Path(c).exists():
        KAGGLE_MODEL_PATH = c
        break

# If static paths fail, scan /kaggle/input recursively for config.json (HF model marker)
if KAGGLE_MODEL_PATH is None:
    print("Static model paths not found. Scanning /kaggle/input/ ...")
    for cfg in sorted(Path("/kaggle/input").rglob("config.json")):
        candidate = str(cfg.parent)
        print(f"  Found model candidate: {candidate}")
        KAGGLE_MODEL_PATH = candidate
        break  # use the first one found

if KAGGLE_MODEL_PATH is None:
    # Debug: show everything in /kaggle/input
    print("DEBUG: Full tree of /kaggle/input/:")
    for p in sorted(Path("/kaggle/input").rglob("*")):
        print(f"  {p}")
    raise FileNotFoundError(
        "No model found in /kaggle/input/. "
        "Ensure the Qwen2 model is attached via kernel-metadata.json model_sources."
    )

SMOKE_SIZE = 10
CONFIDENCE_THRESHOLD = 0.55
OUTPUT_BASE = "/kaggle/working/psm_results"

os.environ["PSM_LLM_BACKEND"]       = "kaggle"
os.environ["PSM_KAGGLE_MODEL_PATH"] = KAGGLE_MODEL_PATH
os.environ["PSM_NUM_PREDICT"]       = "64"
os.environ["PSM_TEMPERATURE"]       = "0.1"

print(f"Model path   : {KAGGLE_MODEL_PATH}")
print(f"Output base  : {OUTPUT_BASE}")
print("✓ Model path verified and configuration applied.")


## 3 · Import PSM modules

In [ ]:
import sys
from pathlib import Path

PSM_ROOT = None
candidates = [
    Path("/kaggle/working/psm_repo/PSM_Experiment_Setup_Research"),
    Path("/kaggle/working/psm_repo/PSM_Experiment_Research_Setup_Updated-main/PSM_Experiment_Setup_Research"),
    Path("/kaggle/input/datasets/kausikvaibhavpatra/psm-experiment/PSM_Experiment_Setup_Research"),
    Path("/kaggle/input/psm-experiment-research-setup-updated/PSM_Experiment_Setup_Research"),
    Path("/kaggle/input/psm-experiment/PSM_Experiment_Setup_Research"),
    Path.cwd() / "PSM_Experiment_Setup_Research",
    Path.cwd(),
]
for candidate in candidates:
    if (candidate / "router").exists() and (candidate / "llm").exists():
        PSM_ROOT = candidate
        break

if PSM_ROOT is None:
    raise RuntimeError("PSM source tree not found.")

sys.path.insert(0, str(PSM_ROOT))
print(f"PSM root: {PSM_ROOT}")

from router.router import Router
from evaluation.metrics import exact_match, token_f1, rouge_l, bleu, contains_any_answer
from run_psm_experiments import PSMExperimentRunner, ExperimentProfile
from triviaqa_smoke_run_fast import PSMSmokeRunFast
print("✓ All PSM modules imported successfully.")


## 4 · Smoke test (synthetic questions)

In [ ]:
import os, shutil
from pathlib import Path
import pandas as pd

SMOKE_OUT = Path(OUTPUT_BASE) / "smoke_test"
shutil.rmtree(SMOKE_OUT, ignore_errors=True)
original_cwd = os.getcwd()
os.makedirs(SMOKE_OUT, exist_ok=True)
os.chdir(SMOKE_OUT)
try:
    smoke = PSMSmokeRunFast(smoke_size=SMOKE_SIZE, run_id="kaggle_smoke")
    smoke_metrics = smoke.run()
finally:
    os.chdir(original_cwd)

if smoke_metrics is None:
    raise RuntimeError("Smoke test produced no metrics.")

print("\n" + "=" * 60)
print("SMOKE TEST RESULT")
print("=" * 60)
print(f"  Total questions : {smoke_metrics['total_queries']}")
print(f"  Avg EM          : {smoke_metrics['avg_em']:.3f}")
print(f"  Avg F1          : {smoke_metrics['avg_f1']:.3f}")
print(f"  Avg Confidence  : {smoke_metrics['avg_confidence']:.3f}")
print("=" * 60)
print("✓ SMOKE TEST PASSED")


### 4a · Smoke test results (CSV preview)

In [ ]:
import glob, pandas as pd
smoke_pred_files = glob.glob(str(SMOKE_OUT / "**" / "predictions.jsonl"), recursive=True)
if smoke_pred_files:
    smoke_df = pd.read_json(smoke_pred_files[0], lines=True)
    print(f"Predictions shape: {smoke_df.shape}")
    display(smoke_df[["query_id", "question", "generated_answer", "correct_answer",
                       "em", "f1", "confidence", "source"]].head(SMOKE_SIZE))
else:
    print("No predictions file found.")


## 5 · Full experiment — TriviaQA

In [ ]:
import os, shutil
from pathlib import Path

EXP_OUT = Path(OUTPUT_BASE) / "experiment_runs"
shutil.rmtree(EXP_OUT, ignore_errors=True)
EXP_OUT.mkdir(parents=True, exist_ok=True)
original_cwd = os.getcwd()
os.chdir(Path(OUTPUT_BASE))
try:
    runner = PSMExperimentRunner(base_results_dir=str(EXP_OUT), threshold=CONFIDENCE_THRESHOLD)
    experiment_results = runner.execute()
finally:
    os.chdir(original_cwd)

print("\n" + "=" * 60)
print("EXPERIMENT COMPLETE")
print(f"  Runs completed : {len(experiment_results['runs'])}")
print(f"  Output dir     : {EXP_OUT}")
print("=" * 60)


## 6 · Evaluation metrics

In [ ]:
import glob, pandas as pd
from pathlib import Path
EXP_OUT = Path(OUTPUT_BASE) / "experiment_runs"
pred_csvs = sorted(EXP_OUT.glob("**/predictions.csv"))
print(f"Found {len(pred_csvs)} predictions.csv file(s)\n")
all_preds = []
for p in pred_csvs:
    run_name = p.parent.name
    df = pd.read_csv(p)
    df["run_name"] = run_name
    all_preds.append(df)
    print(f"─── {run_name}  ({len(df)} rows) ───")
    print(df[["query_id","question","generated_answer","correct_answer",
              "em","f1","rouge_l","bleu","confidence","source",
              "latency_s"]].to_string(index=False, max_colwidth=55))
    print()
if all_preds:
    combined = pd.concat(all_preds, ignore_index=True)
    combined.to_csv(EXP_OUT / "all_predictions_combined.csv", index=False)
    print("✓ Combined predictions saved")


In [ ]:
summary_csvs = sorted(EXP_OUT.glob("**/summary_metrics.csv"))
print(f"Found {len(summary_csvs)} summary_metrics.csv file(s)\n")
all_summaries = []
for s in summary_csvs:
    df = pd.read_csv(s)
    all_summaries.append(df)
if all_summaries:
    summary_df = pd.concat(all_summaries, ignore_index=True)
    cols = ["run_id","num_questions","avg_em","avg_f1","avg_rouge_l","avg_bleu",
            "avg_confidence","avg_latency_s","hallucination_rate",
            "memory_path_rate","retrieval_path_rate","final_memory_size"]
    print(summary_df[cols].to_string(index=False))
master_csv = EXP_OUT / "all_runs_metrics.csv"
if master_csv.exists():
    print("\n─── all_runs_metrics.csv (aggregate) ───")
    print(pd.read_csv(master_csv).to_string(index=False))


## 7 · Output files

In [ ]:
from pathlib import Path
out = Path(OUTPUT_BASE)
print("=" * 70)
print("ALL OUTPUT FILES")
print("=" * 70)
for ext in ["*.csv", "*.json", "*.txt"]:
    files = sorted(out.rglob(ext))
    print(f"\n{ext} ({len(files)} files):")
    for f in files:
        print(f"  {f.relative_to(out)}  ({f.stat().st_size/1024:.1f} KB)")
print("=" * 70)
